# Sandwich Attack Intent Evaluator — Stage 1: Feature Extraction

This notebook extracts raw sandwich data from ClickHouse, computes structural / economic / behavioral features,
and outputs a feature matrix ready for the scoring model.

**Pipeline:**
1. Connect to ClickHouse and load sandwich + sandwich_txs tables
2. Compute per-sandwich **structural features** (consecutive, perfect, cross_block, …)
3. Compute per-sandwich **economic features** (profitA, fee efficiency, …)
4. Identify attackers via Union-Find and compute per-signer **behavioral features**
5. Merge all features into a single feature matrix and export

In [1]:
import os
import hashlib
from collections import defaultdict

import clickhouse_connect
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Load environment
load_dotenv(dotenv_path=".env")
if not os.getenv("CLICKHOUSE_HOST"):
    # Fallback: try parent analyst dir or defaults
    load_dotenv(dotenv_path="../analyst/.env")

CH_HOST = os.getenv("CLICKHOUSE_HOST", os.getenv("NEW_CLICKHOUSE_HOST", "localhost"))
CH_PORT = int(os.getenv("CLICKHOUSE_PORT", os.getenv("NEW_CLICKHOUSE_PORT", "8123")))
CH_USER = os.getenv("CLICKHOUSE_USERNAME", os.getenv("NEW_CLICKHOUSE_USERNAME", "default"))
CH_PASS = os.getenv("CLICKHOUSE_PASSWORD", os.getenv("NEW_CLICKHOUSE_PASSWORD", "sol"))
CH_DB   = os.getenv("CLICKHOUSE_DATABASE", "solwich")

print(f"Connecting to ClickHouse at {CH_HOST}:{CH_PORT}, database={CH_DB}")

Connecting to ClickHouse at localhost:8123, database=solwich


/home/ubuntu/uncover-Solana-sandwich/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
client = clickhouse_connect.get_client(
    host=CH_HOST, port=CH_PORT,
    username=CH_USER, password=CH_PASS,
    database=CH_DB,
)

# Quick sanity check
res = client.query("SELECT count() FROM solwich.sandwiches")
sandwich_count = res.result_rows[0][0]
res2 = client.query("SELECT count() FROM solwich.sandwich_txs")
tx_count = res2.result_rows[0][0]
res3 = client.query("SELECT min(slot), max(slot) FROM solwich.sandwiches")
slot_min, slot_max = res3.result_rows[0]

print(f"Sandwiches: {sandwich_count:,}")
print(f"Sandwich txs: {tx_count:,}")
print(f"Slot range: {slot_min:,} — {slot_max:,}")

Sandwiches: 49,087
Sandwich txs: 490,732
Slot range: 405,462,544 — 405,847,897


## 1. Load Raw Data

We pull two tables:
- `sandwiches` — one row per detected sandwich (structural flags, profit)
- `sandwich_txs` — one row per transaction within a sandwich (front/back/victim/transfer/adverse)

In [3]:
def query_to_df(query: str) -> pd.DataFrame:
    """Execute a ClickHouse query and return a pandas DataFrame."""
    res = client.query(query)
    return pd.DataFrame(res.result_rows, columns=res.column_names)

# ---- Load sandwiches (deduplicated by sandwichId) ----
df_sandwiches = query_to_df("""
    SELECT *
    FROM solwich.sandwiches
    ORDER BY slot, timestamp, sandwichId
    LIMIT 1 BY sandwichId
""")

print(f"Loaded {len(df_sandwiches):,} sandwiches")
print(f"Columns: {list(df_sandwiches.columns)}")
df_sandwiches.head(3)

Loaded 48,220 sandwiches
Columns: ['sandwichId', 'crossBlock', 'slot', 'timestamp', 'tokenA', 'tokenB', 'consecutive', 'multiFrontRun', 'multiBackRun', 'multiVictim', 'frontConsecutive', 'backConsecutive', 'victimConsecutive', 'frontCount', 'backCount', 'victimCount', 'signerSame', 'ownerSame', 'ataSame', 'perfect', 'relativeDiffB', 'profitA']


,sandwichId,crossBlock,slot,timestamp,tokenA,tokenB,consecutive,multiFrontRun,multiBackRun,multiVictim,...,victimConsecutive,frontCount,backCount,victimCount,signerSame,ownerSame,ataSame,perfect,relativeDiffB,profitA
0,54e64aecb8aa4fa15742db3c63608f5a036de02d9f201b...,True,405462544,2026-03-10 11:21:53,SOL,6RMYtdu3p4o4Uo9Ms24p7Gyb1E7NspTXUno1vXA9pump,False,False,False,True,...,False,1,1,6,True,True,False,True,0.000000,0.011490
1,791498de59a612a565dcbbbee7d9e67c8d409e5219bc2c...,True,405465736,2026-03-10 11:42:29,EoutpMzmAjQs9TxmMshMwECXWE4dbuGdsNBZ2KZgkgnp,SOL,False,False,False,True,...,False,1,1,3,True,True,False,False,0.024959,-10343.029369
2,0900132c20bff204ebe8bf9333ec58330fe3cf4af6e25a...,True,405465757,2026-03-10 11:42:38,EoutpMzmAjQs9TxmMshMwECXWE4dbuGdsNBZ2KZgkgnp,SOL,False,False,False,True,...,False,1,1,6,True,True,False,False,0.056799,-22420.223148


In [4]:
# ---- Load sandwich transactions (deduplicated) ----
df_txs = query_to_df("""
    SELECT *
    FROM solwich.sandwich_txs
    ORDER BY sandwichId, slot, position
    LIMIT 1 BY sandwichId, type, signature
""")

print(f"Loaded {len(df_txs):,} sandwich transactions")
print(f"Columns: {list(df_txs.columns)}")
print(f"Type distribution:\n{df_txs['type'].value_counts()}")
df_txs.head(3)

Loaded 481,183 sandwich transactions
Columns: ['sandwichId', 'sandwichTimestamp', 'type', 'slot', 'position', 'timestamp', 'fee', 'signature', 'signers', 'inBundle', 'accountKeys', 'programs', 'fromToken', 'toToken', 'fromAmount', 'toAmount', 'fromTotalAmount', 'toTotalAmount', 'diffA', 'diffB', 'attackerPreBalanceB', 'attackerPostBalanceB', 'poolPreBalanceB', 'poolPostBalanceB', 'ownersOfB']
Type distribution:
type
victim      232998
adverse     150491
backRun      48854
frontRun     48838
transfer         2
Name: count, dtype: int64


,sandwichId,sandwichTimestamp,type,slot,position,timestamp,fee,signature,signers,inBundle,...,toAmount,fromTotalAmount,toTotalAmount,diffA,diffB,attackerPreBalanceB,attackerPostBalanceB,poolPreBalanceB,poolPostBalanceB,ownersOfB
0,00012dd9c3d91275a1cc9c368311a6296274381243b380...,2026-03-11 14:35:13,frontRun,405713868,174,2026-03-11 14:35:13,1105000,rQ3x5pMrP7jdzwuFwBaAWUCD91MwFqwZ77hWie8k9c1GzE...,[3Cfh1tuQNS9tUYernc5uv6AAztRniRysgdw2ZLpbw1VE],False,...,5.242386e+06,0.395062,5.242386e+06,0.0,0.0,0.0,5.242386e+06,5.831970e+08,5.779547e+08,[3Cfh1tuQNS9tUYernc5uv6AAztRniRysgdw2ZLpbw1VE]
1,00012dd9c3d91275a1cc9c368311a6296274381243b380...,2026-03-11 14:35:13,adverse,405713868,308,2026-03-11 14:35:13,25000,bLMGrsed5UGXgjLv9KQXtyFGmsWw9rftGz6ohaar8bhECW...,[fNrJmJ1aQMx1vgnGwJcLWkUCBrDy7GF7ZpVqiXuRFrJ],False,...,6.790902e-01,0.000000,0.000000e+00,0.0,0.0,0.0,0.000000e+00,5.779547e+08,5.870185e+08,[]
2,00012dd9c3d91275a1cc9c368311a6296274381243b380...,2026-03-11 14:35:13,adverse,405713868,311,2026-03-11 14:35:13,25000,5QJypvn6eAo5qxj6f6rgzxHtyEeECBdjxobsfXjoGr8pi8...,[8TirstfwDB9qWWPNRkHg7gCY5WRm5Gw17c7xahjFnGGQ],False,...,6.037238e-01,0.000000,0.000000e+00,0.0,0.0,0.0,0.000000e+00,5.870185e+08,5.952911e+08,[]


In [5]:
# ---- Load slot_txs for cross-block distance estimation ----
slot_min_tx = int(df_txs["slot"].min())
slot_max_tx = int(df_txs["slot"].max())

df_slot_txs = query_to_df(f"""
    SELECT slot, txCount
    FROM solwich.slot_txs
    WHERE slot BETWEEN {slot_min_tx} AND {slot_max_tx}
      AND txFetched = true
    ORDER BY slot
""")

slot_txcount_map = dict(zip(df_slot_txs["slot"], df_slot_txs["txCount"]))
print(f"Loaded tx counts for {len(slot_txcount_map):,} slots")

Loaded tx counts for 370,327 slots


## 2. Extract Per-Sandwich Structural Features

These features come directly from each sandwich's structure — no historical context needed.

In [6]:
def extract_structural_features(df_s: pd.DataFrame, df_t: pd.DataFrame, slot_txcount: dict) -> pd.DataFrame:
    """
    Build per-sandwich structural feature rows by joining the sandwiches table
    with aggregated info from the sandwich_txs table.

    Parameters
    ----------
    df_s : sandwiches table (one row per sandwich)
    df_t : sandwich_txs table (multiple rows per sandwich)
    slot_txcount : dict  slot -> txCount for cross-block distance
    """
    rows = []

    # Pre-group txs by sandwichId for O(1) lookup
    grouped = dict(list(df_t.groupby("sandwichId")))

    for _, s in tqdm(df_s.iterrows(), total=len(df_s), desc="Extracting structural features"):
        sid = s["sandwichId"]
        txs = grouped.get(sid)
        if txs is None or txs.empty:
            continue

        cross_block = bool(s["crossBlock"])

        # Split by type
        fr = txs[txs["type"] == "frontRun"]
        br = txs[txs["type"] == "backRun"]
        victim = txs[txs["type"] == "victim"]
        transfer = txs[txs["type"] == "transfer"]
        adverse = txs[txs["type"] == "adverse"]

        fr_count = len(fr)
        br_count = len(br)
        victim_count = len(victim)
        transfer_count = len(transfer)
        adverse_count = len(adverse)

        if fr_count == 0 or br_count == 0:
            continue

        # --- Positional features ---
        last_front_pos = int(fr["position"].max())
        first_back_pos = int(br["position"].min())
        last_front_slot = int(fr["slot"].max())
        first_back_slot = int(br["slot"].min())

        # In-block distance: positional gap between last front and first back
        if not cross_block:
            front_back_gap = first_back_pos - last_front_pos
        else:
            # Cross-block: estimate total tx distance across slots
            cross_slot_gap = first_back_slot - last_front_slot
            inter_slot_txs = sum(
                slot_txcount.get(sl, 0)
                for sl in range(last_front_slot + 1, first_back_slot)
            )
            # Distance = txs in intermediate slots + positional gap in boundary slots
            front_back_gap = inter_slot_txs + first_back_pos + (
                slot_txcount.get(last_front_slot, last_front_pos) - last_front_pos
            )

        # --- Signer analysis ---
        # Extract primary signer (first element of signers array) for each tx
        def get_primary_signer(signers_val):
            if isinstance(signers_val, (list, tuple)) and len(signers_val) > 0:
                return signers_val[0]
            return str(signers_val) if signers_val else None

        fr_signers = set(fr["signers"].apply(get_primary_signer).dropna())
        br_signers = set(br["signers"].apply(get_primary_signer).dropna())

        # --- Bundle status ---
        # inBundle is currently all False; the field is ready for future use
        attacker_txs = pd.concat([fr, br, transfer])
        total_attacker_txs = len(attacker_txs)
        inbundle_count = int(attacker_txs["inBundle"].sum()) if "inBundle" in attacker_txs.columns else 0

        if total_attacker_txs == 0:
            bundle_ratio = 0.0
        else:
            bundle_ratio = inbundle_count / total_attacker_txs

        in_bundle = bundle_ratio == 1.0 and total_attacker_txs > 0

        # --- Fee ---
        fr_fee = int(fr["fee"].sum()) if not fr.empty else 0
        br_fee = int(br["fee"].sum()) if not br.empty else 0
        total_fee = fr_fee + br_fee

        # --- Cross-block specific ---
        cross_slot_gap = int(first_back_slot - last_front_slot) if cross_block else 0

        row = {
            "sandwichId": sid,
            "slot": int(s["slot"]),
            "timestamp": s["timestamp"],

            # ---- Structural features ----
            "cross_block":        int(cross_block),
            "consecutive":        int(bool(s["consecutive"])),
            "front_consecutive":  int(bool(s["frontConsecutive"])),
            "back_consecutive":   int(bool(s["backConsecutive"])),
            "victim_consecutive": int(bool(s["victimConsecutive"])),
            "perfect":            int(bool(s["perfect"])),
            "relative_diff_b":    float(s["relativeDiffB"]),
            "signer_same":        int(bool(s["signerSame"])),
            "owner_same":         int(bool(s["ownerSame"])),
            "has_transfer":       int(transfer_count > 0),
            "has_adverse":        int(adverse_count > 0),
            "in_bundle":          int(in_bundle),
            "bundle_ratio":       bundle_ratio,

            # ---- Component counts ----
            "fr_count":       fr_count,
            "br_count":       br_count,
            "victim_count":   victim_count,
            "transfer_count": transfer_count,
            "adverse_count":  adverse_count,
            "multi_front":    int(fr_count > 1),
            "multi_back":     int(br_count > 1),
            "multi_victim":   int(victim_count > 1),

            # ---- Positional features ----
            "front_back_gap":   front_back_gap,
            "cross_slot_gap":   cross_slot_gap,

            # ---- Fee (in lamports) ----
            "fr_fee":     fr_fee,
            "br_fee":     br_fee,
            "total_fee":  total_fee,

            # ---- Token pair info (for later grouping) ----
            "tokenA": s["tokenA"],
            "tokenB": s["tokenB"],
        }
        rows.append(row)

    return pd.DataFrame(rows)

df_struct = extract_structural_features(df_sandwiches, df_txs, slot_txcount_map)
print(f"Structural features: {df_struct.shape}")
df_struct.head()

Extracting structural features: 100%|██████████| 48220/48220 [01:37<00:00, 495.90it/s]


Structural features: (48220, 31)


,sandwichId,slot,timestamp,cross_block,consecutive,front_consecutive,back_consecutive,victim_consecutive,perfect,relative_diff_b,...,multi_front,multi_back,multi_victim,front_back_gap,cross_slot_gap,fr_fee,br_fee,total_fee,tokenA,tokenB
0,54e64aecb8aa4fa15742db3c63608f5a036de02d9f201b...,405462544,2026-03-10 11:21:53,1,0,1,1,0,1,0.000000,...,0,0,1,3315,3,3105000,5000,3110000,SOL,6RMYtdu3p4o4Uo9Ms24p7Gyb1E7NspTXUno1vXA9pump
1,791498de59a612a565dcbbbee7d9e67c8d409e5219bc2c...,405465736,2026-03-10 11:42:29,1,0,1,1,0,0,0.024959,...,0,0,1,2626,2,5151,5151,10302,EoutpMzmAjQs9TxmMshMwECXWE4dbuGdsNBZ2KZgkgnp,SOL
2,0900132c20bff204ebe8bf9333ec58330fe3cf4af6e25a...,405465757,2026-03-10 11:42:38,1,0,1,1,0,0,0.056799,...,0,0,1,2654,2,5151,5151,10302,EoutpMzmAjQs9TxmMshMwECXWE4dbuGdsNBZ2KZgkgnp,SOL
3,162695be23ae4c4fd7a0922a0395ed175564514251f60d...,405465768,2026-03-10 11:42:42,1,0,1,1,0,1,0.000000,...,0,0,1,3637,3,105000,35000,140000,SOL,4tKbrrVfWRefFhdcEE4khyfcKVsxUn5xZukqHNoBGG2e
4,24885eeb18fb8ae901ad5641d03172a1e3a269543c6b26...,405465768,2026-03-10 11:42:42,1,0,1,1,0,1,0.000000,...,0,0,1,3643,3,105000,35000,140000,SOL,4tKbrrVfWRefFhdcEE4khyfcKVsxUn5xZukqHNoBGG2e


## 3. Extract Per-Sandwich Economic Features

Profit, fee efficiency, and attack sizing relative to pool liquidity.

In [7]:
def extract_economic_features(df_s: pd.DataFrame, df_t: pd.DataFrame) -> pd.DataFrame:
    """
    Compute economic features per sandwich.

    Uses profitA from the sandwiches table and balance/amount data from sandwich_txs.
    """
    grouped = dict(list(df_t.groupby("sandwichId")))
    rows = []

    for _, s in df_s.iterrows():
        sid = s["sandwichId"]
        txs = grouped.get(sid)
        if txs is None or txs.empty:
            continue

        profit_a = float(s["profitA"])

        fr = txs[txs["type"] == "frontRun"]
        br = txs[txs["type"] == "backRun"]

        if fr.empty or br.empty:
            continue

        # Last front tx and last back tx carry the totals
        last_fr = fr.iloc[-1]
        last_br = br.iloc[-1]

        front_from_total = float(last_fr["fromTotalAmount"]) if last_fr["fromTotalAmount"] else 0.0
        front_to_total   = float(last_fr["toTotalAmount"])   if last_fr["toTotalAmount"]   else 0.0
        back_from_total  = float(last_br["fromTotalAmount"])  if last_br["fromTotalAmount"]  else 0.0
        back_to_total    = float(last_br["toTotalAmount"])    if last_br["toTotalAmount"]    else 0.0

        # Fee in SOL (1 SOL = 1e9 lamports)
        fr_fee = float(fr["fee"].sum())
        br_fee = float(br["fee"].sum())
        total_fee_sol = (fr_fee + br_fee) / 1e9

        # Profit ratio: profit / investment
        profit_ratio = (profit_a / front_from_total) if front_from_total > 0 else 0.0

        # Fee efficiency: profit / total fee (in token A units; compare with fee in SOL only when tokenA == SOL)
        fee_efficiency = (profit_a / total_fee_sol) if total_fee_sol > 0 else 0.0

        # Attack size relative to pool liquidity (using first front tx pool balance)
        pool_pre_b = float(fr.iloc[0]["poolPreBalanceB"]) if fr.iloc[0]["poolPreBalanceB"] else 0.0
        attack_size_ratio = (front_to_total / pool_pre_b) if pool_pre_b > 0 else 0.0

        # DiffA and DiffB from the last back-run tx
        diff_a = float(last_br["diffA"]) if last_br["diffA"] else 0.0
        diff_b = float(last_br["diffB"]) if last_br["diffB"] else 0.0

        rows.append({
            "sandwichId":        sid,
            "profit_a":          profit_a,
            "profit_positive":   int(profit_a > 0),
            "profit_ratio":      np.clip(profit_ratio, -10, 10),   # clip extreme outliers
            "fee_efficiency":    np.clip(fee_efficiency, -1000, 1000),
            "attack_size_ratio": np.clip(attack_size_ratio, 0, 10),
            "total_fee_sol":     total_fee_sol,
            "diff_a":            diff_a,
            "diff_b":            diff_b,
            "front_from_total":  front_from_total,
            "front_to_total":    front_to_total,
            "back_from_total":   back_from_total,
            "back_to_total":     back_to_total,
        })

    return pd.DataFrame(rows)

df_econ = extract_economic_features(df_sandwiches, df_txs)
print(f"Economic features: {df_econ.shape}")
df_econ.describe()

Economic features: (48220, 13)


,profit_a,profit_positive,profit_ratio,fee_efficiency,attack_size_ratio,total_fee_sol,diff_a,diff_b,front_from_total,front_to_total,back_from_total,back_to_total
count,4.822000e+04,48220.000000,48220.000000,48220.000000,4.822000e+04,48220.000000,4.822000e+04,4.822000e+04,4.822000e+04,4.822000e+04,4.822000e+04,4.822000e+04
mean,9.993827e+03,0.606035,0.042698,41.957902,1.763103e-02,0.002699,8.537670e+03,1.155204e+04,2.875626e+05,1.017873e+07,1.016718e+07,2.961003e+05
std,9.100403e+05,0.488632,0.249569,615.054380,4.503876e-02,0.022567,8.668170e+05,1.564882e+05,1.912311e+06,3.395785e+07,3.395142e+07,2.171384e+06
min,-7.787494e+07,0.000000,-0.983845,-1000.000000,4.846102e-08,0.000010,-7.878712e+07,-2.422858e+01,1.000000e-04,1.011810e-04,1.092520e-04,1.000020e-04
25%,-1.364226e-02,0.000000,-0.005326,-89.011703,2.317300e-03,0.000015,-2.072037e-02,0.000000e+00,3.346984e-01,6.410665e+02,6.261564e+02,3.382046e-01
50%,4.047245e-03,1.000000,0.009177,10.448077,6.838671e-03,0.000090,5.039053e-03,0.000000e+00,9.560322e-01,1.427131e+06,1.410854e+06,9.422172e-01
75%,8.063989e-02,1.000000,0.066458,272.281555,1.493997e-02,0.000810,9.726194e-02,1.192093e-07,3.614815e+00,7.990139e+06,7.984300e+06,3.826755e+00
max,1.358862e+08,1.000000,10.000000,1000.000000,9.900744e-01,2.501015,1.293053e+08,1.644270e+07,1.538629e+08,9.200366e+08,9.200366e+08,1.649952e+08


## 4. Identify Attackers (Union-Find) and Extract Behavioral Features

Group signers into attacker identities using Union-Find on shared sandwiches,
then compute per-attacker behavioral statistics and map them back to each sandwich.

In [8]:
# ---------- Union-Find for attacker clustering ----------

class UnionFind:
    """Disjoint-set (Union-Find) with path compression and union by rank."""

    def __init__(self):
        self.parent = {}
        self.rank = {}

    def find(self, x):
        self.parent.setdefault(x, x)
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        self.rank.setdefault(ra, 0)
        self.rank.setdefault(rb, 0)
        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1

In [9]:
ATTACKER_TX_TYPES = ["frontRun", "backRun", "transfer"]


def get_primary_signer(signers_val):
    """Extract the first signer from a signers array value."""
    if isinstance(signers_val, (list, tuple)) and len(signers_val) > 0:
        return str(signers_val[0])
    if isinstance(signers_val, str) and signers_val:
        return signers_val
    return None


def build_attacker_mapping(df_t: pd.DataFrame) -> tuple[dict, dict]:
    """
    Cluster signers into attacker identities.

    Returns
    -------
    sandwich_to_attacker : dict  sandwichId -> attacker_key
    attacker_members     : dict  attacker_key -> set of signer addresses
    """
    uf = UnionFind()

    attacker_txs = df_t[df_t["type"].isin(ATTACKER_TX_TYPES)].copy()
    attacker_txs["primary_signer"] = attacker_txs["signers"].apply(get_primary_signer)

    # Union signers that appear together in the same sandwich's attacker transactions
    for _sid, grp in attacker_txs.groupby("sandwichId"):
        signers = sorted(grp["primary_signer"].dropna().unique())
        if len(signers) <= 1:
            if len(signers) == 1:
                uf.find(signers[0])  # register singleton
            continue
        for s in signers[1:]:
            uf.union(signers[0], s)

    # Build component membership
    comp_members = defaultdict(set)
    all_signers = set(attacker_txs["primary_signer"].dropna().unique())
    for s in all_signers:
        comp_members[uf.find(s)].add(s)

    # Deterministic attacker key from sorted member addresses
    root_to_key = {}
    for root, members in comp_members.items():
        key_str = ",".join(sorted(members))
        root_to_key[root] = hashlib.sha256(key_str.encode()).hexdigest()[:16]

    signer_to_key = {s: root_to_key[uf.find(s)] for s in all_signers}

    # Map each sandwich to its attacker
    sandwich_to_attacker = {}
    for sid, grp in attacker_txs.groupby("sandwichId"):
        signers = grp["primary_signer"].dropna().unique()
        keys = sorted({signer_to_key[s] for s in signers if s in signer_to_key})
        sandwich_to_attacker[sid] = keys[0] if len(keys) == 1 else (keys[0] if keys else "UNKNOWN")

    attacker_members = {root_to_key[root]: members for root, members in comp_members.items()}

    return sandwich_to_attacker, attacker_members


sandwich_to_attacker, attacker_members = build_attacker_mapping(df_txs)
print(f"Identified {len(attacker_members):,} unique attacker entities")
print(f"Mapped {len(sandwich_to_attacker):,} sandwiches to attackers")

# Show top attackers by sandwich count
attacker_counts = pd.Series(sandwich_to_attacker).value_counts()
print(f"\nTop 10 attackers by sandwich count:")
for ak, cnt in attacker_counts.head(10).items():
    n_addrs = len(attacker_members.get(ak, set()))
    print(f"  {ak}: {cnt:,} sandwiches, {n_addrs} address(es)")

Identified 10,782 unique attacker entities
Mapped 48,221 sandwiches to attackers

Top 10 attackers by sandwich count:
  bcc905d5cc521f2e: 523 sandwiches, 127 address(es)
  df3ffa06724c89f9: 494 sandwiches, 1 address(es)
  81f0f12125ba58bc: 420 sandwiches, 1 address(es)
  5fe1ae8d708134b6: 400 sandwiches, 1 address(es)
  3d091f79199cec46: 385 sandwiches, 1 address(es)
  09282e8a064d97c6: 375 sandwiches, 1 address(es)
  a5192204be763525: 360 sandwiches, 1 address(es)
  f282c2da7bc67953: 358 sandwiches, 1 address(es)
  7ac9650fa8a91dc0: 346 sandwiches, 1 address(es)
  9eab6a2589e3ed6a: 343 sandwiches, 1 address(es)


In [10]:
def compute_behavioral_features(
    df_s: pd.DataFrame,
    sandwich_to_attacker: dict,
) -> pd.DataFrame:
    """
    Compute per-attacker behavioral statistics, then map back to each sandwich.

    For each sandwich, the behavioral features describe the attacker's *historical*
    pattern at the time of that sandwich. To avoid label leakage, we use
    cumulative stats up to (but not including) the current sandwich.

    For the initial batch (offline) run we use the full-period aggregation as
    a simplification — the online scoring pipeline will use true temporal splits.
    """
    # Attach attacker key to sandwiches
    df = df_s[["sandwichId", "slot", "crossBlock", "consecutive",
               "signerSame", "ownerSame", "perfect", "profitA", "relativeDiffB"]].copy()
    df["attacker_key"] = df["sandwichId"].map(sandwich_to_attacker)
    df["win"] = (df["profitA"] > 0).astype(int)

    # ---------- Aggregate per-attacker stats (full period) ----------
    agg = df.groupby("attacker_key").agg(
        signer_sandwich_count  = ("sandwichId", "nunique"),
        signer_win_count       = ("win", "sum"),
        signer_avg_profit      = ("profitA", "mean"),
        signer_total_profit    = ("profitA", "sum"),
        signer_profit_std      = ("profitA", "std"),
        signer_cross_block_sum = ("crossBlock", "sum"),
        signer_consec_sum      = ("consecutive", "sum"),
        signer_perfect_sum     = ("perfect", "sum"),
        signer_signer_same_sum = ("signerSame", "sum"),
        signer_owner_same_sum  = ("ownerSame", "sum"),
        signer_min_slot        = ("slot", "min"),
        signer_max_slot        = ("slot", "max"),
    ).reset_index()

    # Derived ratios
    agg["signer_win_rate"]         = agg["signer_win_count"] / agg["signer_sandwich_count"]
    agg["signer_cross_block_rate"] = agg["signer_cross_block_sum"] / agg["signer_sandwich_count"]
    agg["signer_consec_rate"]      = agg["signer_consec_sum"] / agg["signer_sandwich_count"]
    agg["signer_perfect_rate"]     = agg["signer_perfect_sum"] / agg["signer_sandwich_count"]
    agg["signer_same_rate"]        = agg["signer_signer_same_sum"] / agg["signer_sandwich_count"]
    agg["signer_span_slots"]       = agg["signer_max_slot"] - agg["signer_min_slot"]

    # Temporal density (sandwiches per 1000 slots)
    agg["signer_temporal_density"] = np.where(
        agg["signer_span_slots"] > 0,
        agg["signer_sandwich_count"] / (agg["signer_span_slots"] / 1000),
        0.0,
    )

    # Pool diversity: count unique tokenB per attacker
    pool_div = df_s[["sandwichId", "tokenB"]].copy()
    pool_div["attacker_key"] = pool_div["sandwichId"].map(sandwich_to_attacker)
    pool_counts = pool_div.groupby("attacker_key")["tokenB"].nunique().reset_index()
    pool_counts.columns = ["attacker_key", "signer_pool_diversity"]
    agg = agg.merge(pool_counts, on="attacker_key", how="left")
    agg["signer_pool_diversity"] = agg["signer_pool_diversity"].fillna(0).astype(int)

    # Number of signer addresses in this attacker entity
    agg["signer_address_count"] = agg["attacker_key"].map(
        lambda k: len(attacker_members.get(k, set()))
    )

    agg["signer_profit_std"] = agg["signer_profit_std"].fillna(0.0)

    # Select behavioral feature columns
    behav_cols = [
        "attacker_key",
        "signer_sandwich_count",
        "signer_win_rate",
        "signer_avg_profit",
        "signer_total_profit",
        "signer_profit_std",
        "signer_cross_block_rate",
        "signer_consec_rate",
        "signer_perfect_rate",
        "signer_same_rate",
        "signer_temporal_density",
        "signer_pool_diversity",
        "signer_address_count",
        "signer_span_slots",
    ]

    # Map back to sandwich level
    sandwich_attacker = pd.DataFrame({
        "sandwichId": list(sandwich_to_attacker.keys()),
        "attacker_key": list(sandwich_to_attacker.values()),
    })
    df_behav = sandwich_attacker.merge(agg[behav_cols], on="attacker_key", how="left")

    return df_behav


df_behav = compute_behavioral_features(df_sandwiches, sandwich_to_attacker)
print(f"Behavioral features: {df_behav.shape}")
df_behav.head()

Behavioral features: (48221, 15)


,sandwichId,attacker_key,signer_sandwich_count,signer_win_rate,signer_avg_profit,signer_total_profit,signer_profit_std,signer_cross_block_rate,signer_consec_rate,signer_perfect_rate,signer_same_rate,signer_temporal_density,signer_pool_diversity,signer_address_count,signer_span_slots
0,00012dd9c3d91275a1cc9c368311a6296274381243b380...,03e1d26b0ee6b80c,16.0,0.812500,-0.010907,-0.174516,0.076461,0.812500,0.000000,1.0,1.0,0.122897,16.0,1.0,130190.0
1,000143017b1c38b3a6da1db6d3dd7cc68ec58b939fd6cf...,81f0f12125ba58bc,420.0,0.907143,0.103435,43.442572,0.176689,0.530952,0.002381,1.0,1.0,1.116190,420.0,1.0,376280.0
2,00030251c86cdb7c8f7c9892f09b099eb1689034bbd6be...,c866416ea471c485,4.0,0.250000,-5226.958215,-20907.832859,7901.929004,0.750000,0.000000,0.0,1.0,0.027116,2.0,1.0,147512.0
3,0005a3b7ffeae2debd291acf40dcd4d004b8dcae6fb489...,8536f08671b04496,4.0,1.000000,0.000094,0.000375,0.000056,1.000000,0.000000,1.0,1.0,0.020140,4.0,1.0,198608.0
4,00078aa09d994e7ffe8c84a1a78b64b715a5863312b0d5...,268bd78135e85685,4.0,0.250000,-0.040209,-0.160835,0.058855,1.000000,0.000000,1.0,1.0,0.044663,4.0,1.0,89560.0


## 5. Merge All Features into Final Feature Matrix

In [11]:
# Merge structural + economic + behavioral
df_features = df_struct.merge(df_econ, on="sandwichId", how="inner")
df_features = df_features.merge(df_behav, on="sandwichId", how="left")

print(f"Final feature matrix: {df_features.shape}")
print(f"Columns ({len(df_features.columns)}): {list(df_features.columns)}")
print(f"\nMissing values:\n{df_features.isnull().sum()[df_features.isnull().sum() > 0]}")

df_features.head()

Final feature matrix: (48220, 57)
Columns (57): ['sandwichId', 'slot', 'timestamp', 'cross_block', 'consecutive', 'front_consecutive', 'back_consecutive', 'victim_consecutive', 'perfect', 'relative_diff_b', 'signer_same', 'owner_same', 'has_transfer', 'has_adverse', 'in_bundle', 'bundle_ratio', 'fr_count', 'br_count', 'victim_count', 'transfer_count', 'adverse_count', 'multi_front', 'multi_back', 'multi_victim', 'front_back_gap', 'cross_slot_gap', 'fr_fee', 'br_fee', 'total_fee', 'tokenA', 'tokenB', 'profit_a', 'profit_positive', 'profit_ratio', 'fee_efficiency', 'attack_size_ratio', 'total_fee_sol', 'diff_a', 'diff_b', 'front_from_total', 'front_to_total', 'back_from_total', 'back_to_total', 'attacker_key', 'signer_sandwich_count', 'signer_win_rate', 'signer_avg_profit', 'signer_total_profit', 'signer_profit_std', 'signer_cross_block_rate', 'signer_consec_rate', 'signer_perfect_rate', 'signer_same_rate', 'signer_temporal_density', 'signer_pool_diversity', 'signer_address_count', 'sign

,sandwichId,slot,timestamp,cross_block,consecutive,front_consecutive,back_consecutive,victim_consecutive,perfect,relative_diff_b,...,signer_total_profit,signer_profit_std,signer_cross_block_rate,signer_consec_rate,signer_perfect_rate,signer_same_rate,signer_temporal_density,signer_pool_diversity,signer_address_count,signer_span_slots
0,54e64aecb8aa4fa15742db3c63608f5a036de02d9f201b...,405462544,2026-03-10 11:21:53,1,0,1,1,0,1,0.000000,...,0.762165,0.057971,0.811111,0.0,0.944444,1.0,0.256568,90.0,1.0,350784.0
1,791498de59a612a565dcbbbee7d9e67c8d409e5219bc2c...,405465736,2026-03-10 11:42:29,1,0,1,1,0,0,0.024959,...,73173.828367,51700.586946,1.000000,0.0,0.000000,1.0,0.024525,2.0,1.0,163096.0
2,0900132c20bff204ebe8bf9333ec58330fe3cf4af6e25a...,405465757,2026-03-10 11:42:38,1,0,1,1,0,0,0.056799,...,-45961.149135,11998.894497,1.000000,0.0,0.000000,1.0,0.036921,2.0,1.0,162511.0
3,162695be23ae4c4fd7a0922a0395ed175564514251f60d...,405465768,2026-03-10 11:42:42,1,0,1,1,0,1,0.000000,...,-110001.874711,33222.597901,0.965318,0.0,0.994220,1.0,0.917188,286.0,1.0,377240.0
4,24885eeb18fb8ae901ad5641d03172a1e3a269543c6b26...,405465768,2026-03-10 11:42:42,1,0,1,1,0,1,0.000000,...,-0.986385,0.054660,0.954268,0.0,1.000000,1.0,0.866549,280.0,1.0,378513.0


In [12]:
# Fill NaN behavioral features for sandwiches with unknown attackers
behav_numeric_cols = [
    "signer_sandwich_count", "signer_win_rate", "signer_avg_profit",
    "signer_total_profit", "signer_profit_std", "signer_cross_block_rate",
    "signer_consec_rate", "signer_perfect_rate", "signer_same_rate",
    "signer_temporal_density", "signer_pool_diversity",
    "signer_address_count", "signer_span_slots",
]

for col in behav_numeric_cols:
    if col in df_features.columns:
        df_features[col] = df_features[col].fillna(0)

if "attacker_key" in df_features.columns:
    df_features["attacker_key"] = df_features["attacker_key"].fillna("UNKNOWN")

print(f"Missing values after fillna: {df_features.isnull().sum().sum()}")

Missing values after fillna: 0


## 6. Feature Overview and Sanity Checks

In [13]:
print("=" * 60)
print("FEATURE MATRIX SUMMARY")
print("=" * 60)

# Data shape
print(f"\nRows: {len(df_features):,}")
print(f"Columns: {len(df_features.columns)}")

# Cross-block vs in-block distribution
cb_count = df_features["cross_block"].sum()
ib_count = len(df_features) - cb_count
print(f"\nIn-block sandwiches:    {ib_count:,} ({ib_count/len(df_features)*100:.1f}%)")
print(f"Cross-block sandwiches: {cb_count:,} ({cb_count/len(df_features)*100:.1f}%)")

# Key feature distributions
for col in ["signer_same", "owner_same", "consecutive", "perfect", "in_bundle", "has_transfer", "has_adverse", "profit_positive"]:
    if col in df_features.columns:
        pct = df_features[col].mean() * 100
        print(f"{col:25s}: {pct:6.2f}% = 1")

print(f"\n--- Numeric feature stats ---")
numeric_cols = ["relative_diff_b", "front_back_gap", "profit_a", "profit_ratio",
               "fee_efficiency", "attack_size_ratio", "victim_count",
               "signer_sandwich_count", "signer_win_rate", "signer_temporal_density",
               "signer_pool_diversity"]
for col in numeric_cols:
    if col in df_features.columns:
        s = df_features[col]
        print(f"{col:30s}: mean={s.mean():12.4f}  median={s.median():12.4f}  std={s.std():12.4f}")

FEATURE MATRIX SUMMARY

Rows: 48,220
Columns: 57

In-block sandwiches:    5,484 (11.4%)
Cross-block sandwiches: 42,736 (88.6%)
signer_same              :  99.39% = 1
owner_same               :  96.26% = 1
consecutive              :   0.82% = 1
perfect                  :  68.66% = 1
in_bundle                :   0.00% = 1
has_transfer             :   0.00% = 1
has_adverse              :  71.12% = 1
profit_positive          :  60.60% = 1

--- Numeric feature stats ---
relative_diff_b               : mean=      0.0143  median=      0.0000  std=      0.0271
front_back_gap                : mean=   2749.5959  median=   2731.0000  std=   1588.2807
profit_a                      : mean=   9993.8272  median=      0.0040  std= 910040.3243
profit_ratio                  : mean=      0.0427  median=      0.0092  std=      0.2496
fee_efficiency                : mean=     41.9579  median=     10.4481  std=    615.0544
attack_size_ratio             : mean=      0.0176  median=      0.0068  std=      0.0

## 7. Export Feature Matrix

In [14]:
# Export to parquet (efficient, typed) and CSV (human-readable)
os.makedirs("data", exist_ok=True)

OUT_PARQUET = "data/features.parquet"
OUT_CSV = "data/features.csv"

df_features.to_parquet(OUT_PARQUET, index=False, engine="pyarrow", compression="zstd")
print(f"Saved {OUT_PARQUET} ({os.path.getsize(OUT_PARQUET) / 1024:.0f} KB)")

df_features.to_csv(OUT_CSV, index=False)
print(f"Saved {OUT_CSV} ({os.path.getsize(OUT_CSV) / 1024:.0f} KB)")

# Also export the attacker mapping for later use
attacker_map_rows = []
for ak, members in attacker_members.items():
    for addr in sorted(members):
        attacker_map_rows.append({"attacker_key": ak, "signer_address": addr})
df_attacker_map = pd.DataFrame(attacker_map_rows)
df_attacker_map.to_csv("data/attacker_mapping.csv", index=False)
print(f"Saved attacker mapping: {len(df_attacker_map)} rows, {df_attacker_map['attacker_key'].nunique()} attackers")

Saved data/features.parquet (7880 KB)
Saved data/features.csv (26465 KB)
Saved attacker mapping: 11269 rows, 10782 attackers


In [15]:
# ---- Define the feature columns that will be used by the scoring model ----

# Structural features (available at detection time, no history needed)
STRUCTURAL_FEATURES = [
    "cross_block",
    "consecutive",
    "front_consecutive",
    "back_consecutive",
    "victim_consecutive",
    "perfect",
    "relative_diff_b",
    "signer_same",
    "owner_same",
    "has_transfer",
    "has_adverse",
    "in_bundle",
    "bundle_ratio",
    "fr_count",
    "br_count",
    "victim_count",
    "multi_front",
    "multi_back",
    "multi_victim",
    "front_back_gap",
    "cross_slot_gap",
]

# Economic features (available at detection time)
ECONOMIC_FEATURES = [
    "profit_a",
    "profit_positive",
    "profit_ratio",
    "fee_efficiency",
    "attack_size_ratio",
    "total_fee_sol",
]

# Behavioral features (require attacker history, cold-start safe via fillna=0)
BEHAVIORAL_FEATURES = [
    "signer_sandwich_count",
    "signer_win_rate",
    "signer_avg_profit",
    "signer_total_profit",
    "signer_profit_std",
    "signer_cross_block_rate",
    "signer_consec_rate",
    "signer_perfect_rate",
    "signer_same_rate",
    "signer_temporal_density",
    "signer_pool_diversity",
    "signer_address_count",
    "signer_span_slots",
]

ALL_FEATURES = STRUCTURAL_FEATURES + ECONOMIC_FEATURES + BEHAVIORAL_FEATURES

# Verify all features exist
missing = [f for f in ALL_FEATURES if f not in df_features.columns]
if missing:
    print(f"WARNING: missing features: {missing}")
else:
    print(f"All {len(ALL_FEATURES)} model features present in the feature matrix.")

print(f"\nStructural:  {len(STRUCTURAL_FEATURES)} features")
print(f"Economic:    {len(ECONOMIC_FEATURES)} features")
print(f"Behavioral:  {len(BEHAVIORAL_FEATURES)} features")
print(f"Total:       {len(ALL_FEATURES)} features")

All 40 model features present in the feature matrix.

Structural:  21 features
Economic:    6 features
Behavioral:  13 features
Total:       40 features
